# WideBind — Colab Training (D=4096, Option C+, T4)
~293M params, 32 layers, PartitionedEmbed/Head (K=32, S=6), GroupedCognitiveMirror (16 experts, k=32, K-space gate), GroupedMLP (16, expand=4), surv=2.0.

K-space gate: |pred_error| per-token per-expert (k=32). W_pred with aux MSE loss, weight_decay=0.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/widebind_data'
print(f'Data path: {DRIVE}')
!ls -lh "$DRIVE"/*.bin 2>/dev/null || echo 'No .bin files found'

In [ ]:
# Locate source files on Drive and copy locally
import sys, os, shutil
SRC_LOCAL = '/content/widebind_src'

src_drive = None
for d in [os.path.join(DRIVE, 'src'), DRIVE]:
    subdirs = ['core', 'compression', 'scripts', 'notebooks']
    if all(os.path.isdir(os.path.join(d, sd)) for sd in subdirs):
        src_drive = d
        break

if src_drive:
    if os.path.isdir(SRC_LOCAL):
        shutil.rmtree(SRC_LOCAL)
    shutil.copytree(src_drive, SRC_LOCAL)
    sys.path.insert(0, SRC_LOCAL)
    sys.path.insert(0, os.path.join(SRC_LOCAL, 'scripts'))
    print(f'Source files copied from {src_drive}')
else:
    print('Directories not found on Drive. Checking files...')
    for d in [os.path.join(DRIVE, 'src'), DRIVE]:
        for sd in ['core', 'compression', 'scripts']:
            p = os.path.join(d, sd)
            exists = os.path.isdir(p)
            print(f'  {p}: {"OK" if exists else "MISSING"}')
    print('\nUpload directories: core/, compression/, scripts/, notebooks/ to Drive')

%cd /content

In [ ]:
# Verify imports
from core import WideBindConfig, WideBindStack, LiveInference, MirrorMonitor
from compression import FCF_CPR

cfg = WideBindConfig(D=4096, n_layers=32, bind_K=32,
                     mlp_groups=16, mlp_expand=4,
                     code_dim=32, code_sparsity=6, mirror_k=32)
model = WideBindStack(cfg)
print(f'Model: {model.param_count():,} params, surv=K/G={cfg.bind_K}/{cfg.mlp_groups}={cfg.bind_K/cfg.mlp_groups:.0f}')

# Quick sanity: check shapes (Option C+: G=16, k=32)
wg = model.layers[0].mirror.w_gate
assert wg.shape == (16, 32), f'w_gate shape {wg.shape} != (16,32) -- wrong config!'
print(f'Gate: w_gate {list(wg.shape)}, surv={cfg.bind_K/cfg.mlp_groups:.0f}')
del model

In [ ]:
# ---- Patch FCF_CPR: sparse_block_codes instead of zeckendorf_codes ----
# Old fcf_cpr.py on Drive uses zeckendorf_codes(K~23); new model uses
# PartitionedEmbed/Head with K=32 sparse_block_codes.
from core import sparse_block_codes, dct_basis
from compression.fcf_cpr import FCF_CPR, dequantize_tensor

def _patched_decompress(self, compressed, meta, cfg):
    import torch
    sd = {}
    V_dct = dct_basis(cfg.D)
    codes = sparse_block_codes(cfg.vocab, K=cfg.code_dim, S=cfg.code_sparsity)
    n_layers = cfg.n_layers
    for k, v in compressed.items():
        info = meta.get(k)
        if info is None:
            sd[k] = v.clone()
        elif info[0] == 'scalar':
            orig_shape, orig_dtype = info[1], info[2]
            if len(orig_shape) > 0 and orig_shape[0] > 1:
                sd[k] = v.detach().expand(orig_shape).clone().to(orig_dtype)
            else:
                sd[k] = v.detach().clone().to(orig_dtype)
        elif info[0] == 'uniform8':
            shape, dtype, t_min, scale = info[1], info[2], info[3], info[4]
            sd[k] = dequantize_tensor(v, t_min, scale, dtype)
    for i in range(n_layers):
        sd[f'layers.{i}.V_dct'] = V_dct.clone()
    sd['embed.codes'] = codes.clone()
    sd['lm_head.codes'] = codes.clone()
    return sd

FCF_CPR.decompress_sd = _patched_decompress
print('FCF_CPR patched: decompress_sd uses sparse_block_codes')


In [ ]:
# ---- Launch training ----
# Config
D            = 4096
N_LAYERS     = 32
BIND_K       = 32
MLP_GROUPS   = 16
MLP_EXPAND   = 4
CODE_DIM     = 32
CODE_SPARSITY = 6
MIRROR_K     = 32
SEQ_LEN      = 64
LR           = 3e-4
MAX_STEPS    = 300000
WARMUP       = 2000
SAVE_EVERY   = 5000
EVAL_EVERY   = 2000
LOG_EVERY    = 100
SCHEDULER    = 'mirror'

print('=' * 60)
print(f'WideBind D={D} G={MLP_GROUPS}d{D//MLP_GROUPS}exp{MLP_EXPAND}')
print(f'  Layers: {N_LAYERS} | K={BIND_K} surv={BIND_K/MLP_GROUPS:.0f} | codes={CODE_DIM}/{CODE_SPARSITY} | mirror_k={MIRROR_K}')
print(f'  LR={LR} | warmup={WARMUP} | steps={MAX_STEPS}')
print(f'  Scheduler: {SCHEDULER}')
print(f'  Save dir: {os.path.join(DRIVE, "checkpoints")}')
print('=' * 60)

In [ ]:
# ---- Training loop (RAM-optimized, W_pred≈I) ----
import os, math, sys, time, glob, gc
import torch
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device} ({torch.cuda.get_device_name(0)})')
mem_total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'VRAM: {mem_total:.1f} GB')

cfg = WideBindConfig(
    D=D, n_layers=N_LAYERS, bind_K=BIND_K,
    mlp_groups=MLP_GROUPS, mlp_expand=MLP_EXPAND,
    code_dim=CODE_DIM, code_sparsity=CODE_SPARSITY, mirror_k=MIRROR_K,
    seq_len=SEQ_LEN, lr=LR, max_steps=MAX_STEPS,
    warmup_steps=WARMUP, scheduler=SCHEDULER,
    data_dir=DRIVE, save_dir=os.path.join(DRIVE, 'checkpoints'),
    log_dir=os.path.join(DRIVE, 'logs'),
    grad_clip=1.0, conv_kernel=48,
)

# --- data ---
print(f'Loading data from {DRIVE}')
stream_files = sorted(glob.glob(os.path.join(DRIVE, 'token_stream_*_clean.bin')))
if not stream_files:
    stream_files = sorted(glob.glob(os.path.join(DRIVE, 'token_stream_*.bin')))
streams = []
for f in stream_files:
    streams.append(np.fromfile(f, dtype=np.uint16))
total_tokens = sum(len(s) for s in streams)
print(f'Found {len(streams)} files, {total_tokens:,} total tokens')

# --- model ---
model = WideBindStack(cfg).to(device)
print(f'Model: {model.param_count():,} params')

# --- batch size ---
for bs in [4, 2, 1]:
    torch.cuda.empty_cache(); gc.collect()
    try:
        x = torch.randint(0, 50000, (bs, SEQ_LEN), device=device)
        h = model.embed_tokens(x); out, state, _ = model(h)
        out[:, :1].sum().backward(); model.zero_grad()
        del x, h, out, state
        cfg.batch_size = bs; print(f'  Batch size {bs}: OK'); break
    except RuntimeError:
        model.zero_grad(); torch.cuda.empty_cache(); gc.collect()
        print(f'  Batch size {bs}: OOM')
print(f'  Using batch size: {cfg.batch_size}')

from core import MirrorLRScheduler, AdaptiveController
param_groups = model.param_groups()
optimizer = torch.optim.AdamW(param_groups, betas=(0.9, 0.95))
scheduler = MirrorLRScheduler(model, optimizer, cfg=cfg)
cpr = FCF_CPR()

# --- resume (simple) — step_10000.pt НЕСОВМЕСТИМ (G=32->16), нужен fresh start ---
start_step = 0; best_val_loss = float('inf')
os.makedirs(cfg.save_dir, exist_ok=True)
def step_key(p):
    name = os.path.basename(p)
    if name == 'best.pt': return 999999
    import re; m = re.search(r'step_(\d+)', name)
    return int(m.group(1)) if m else 0
ckpt_files = sorted(glob.glob(os.path.join(cfg.save_dir, '*.pt')), key=step_key)
if ckpt_files:
    latest = ckpt_files[-1]
    print(f'Resuming from {latest}')
    ckpt = cpr.load_compressed(latest, cfg=cfg)
    missing, _ = model.load_state_dict(ckpt['model'], strict=False)
    if missing: print(f'  Missing keys (fresh init): {len(missing)}')
    if 'optimizer' in ckpt:
        try: optimizer.load_state_dict(ckpt['optimizer'])
        except: pass
    if 'scheduler' in ckpt:
        try: scheduler.load_state_dict(ckpt['scheduler'])
        except: scheduler._step = ckpt.get('step', 0)
    else: scheduler._step = ckpt.get('step', 0)
    start_step = ckpt['step']
    best_val_loss = ckpt.get('best_val_loss', float('inf'))
    print(f'  Resume at step {start_step}')
    del ckpt; gc.collect()
else:
    print('  Fresh start (no checkpoint)')

# --- eval ---
@torch.no_grad()
def evaluate():
    model.eval()
    total_loss = 0.0; total_steps = 0
    n_eval = min(50, sum(len(s) for s in streams) // (cfg.batch_size * SEQ_LEN) // len(streams))
    for si, stream in enumerate(streams):
        off = max(len(stream) // 4, cfg.batch_size * SEQ_LEN + 1)
        state = None; gs = None
        for i in range(n_eval):
            if off + cfg.batch_size * SEQ_LEN + 1 > len(stream): off = 0; break
            chunk = stream[off:off+cfg.batch_size*SEQ_LEN+1]
            x = torch.from_numpy(chunk[:cfg.batch_size*SEQ_LEN].copy()).long().view(cfg.batch_size, SEQ_LEN).to(device)
            y = torch.from_numpy(chunk[1:cfg.batch_size*SEQ_LEN+1].copy()).long().view(cfg.batch_size, SEQ_LEN).to(device)
            off += cfg.batch_size * SEQ_LEN
            h = model.embed_tokens(x); out, state, gs = model(h, state, global_state=gs)
            total_loss += model.compute_loss(out, y).item(); total_steps += 1
            if i % 25 == 24: state = None; gs = None
        del state, gs; gc.collect()
    model.train()
    return total_loss / max(total_steps, 1)

# --- train ---
state = None; global_state = None; stream_idx = 0; offset = 0
tokens_seen = 0; t0 = time.time()

print(f'Training: step {start_step} -> {MAX_STEPS}')
try:
    for step in range(start_step, MAX_STEPS):
        stream = streams[stream_idx]
        if offset + cfg.batch_size * SEQ_LEN + 1 > len(stream):
            offset = 0; stream_idx = (stream_idx + 1) % len(streams)
            stream = streams[stream_idx]; state = None; global_state = None
        chunk = stream[offset:offset+cfg.batch_size*SEQ_LEN+1]
        x = torch.from_numpy(chunk[:cfg.batch_size*SEQ_LEN].copy()).long().view(cfg.batch_size, SEQ_LEN).to(device)
        y = torch.from_numpy(chunk[1:cfg.batch_size*SEQ_LEN+1].copy()).long().view(cfg.batch_size, SEQ_LEN).to(device)
        offset += cfg.batch_size * SEQ_LEN

        h = model.embed_tokens(x); out, state, global_state = model(h, state, global_state=global_state)
        loss = model.compute_loss(out, y)
        del x, y, h, out

        if torch.isnan(loss) or torch.isinf(loss):
            optimizer.zero_grad(set_to_none=True); continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); optimizer.zero_grad(set_to_none=True)
        scheduler.step(); tokens_seen += cfg.batch_size * SEQ_LEN

        if step % LOG_EVERY == 0:
            skip = model.layers[0].mirror.log_skip_alpha.exp().mean().item()
            dt = time.time() - t0
            print(f'  step={step:>6} loss={loss.item():.4f} α_skip={skip:.3f} tok/s={tokens_seen/max(dt,1e-8):.0f}')

        if step > 0 and step % EVAL_EVERY == 0:
            val_loss = evaluate()
            print(f'  EVAL step={step}: val_loss={val_loss:.4f}')
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                cpr.save_compressed({'step': step, 'model': model.state_dict(),
                    'best_val_loss': best_val_loss, 'cfg': cfg},
                    os.path.join(cfg.save_dir, 'best.pt'))
                print(f'  New best!')
            torch.cuda.empty_cache(); gc.collect()

        if step > 0 and step % SAVE_EVERY == 0:
            cpr.save_compressed({'step': step, 'model': model.state_dict(),
                'best_val_loss': best_val_loss, 'cfg': cfg},
                os.path.join(cfg.save_dir, f'step_{step}.pt'))
            print(f'  Checkpoint saved')

except KeyboardInterrupt:
    cpr.save_compressed({'step': step, 'model': model.state_dict(),
        'best_val_loss': best_val_loss, 'cfg': cfg},
        os.path.join(cfg.save_dir, f'interrupt_step_{step}.pt'))
    print(f'\nInterrupted. Saved.')

In [ ]:
# --- Analyze latest checkpoint ---
from scripts.analyze_checkpoint import generate_report
import glob, shutil

latest_ckpt = sorted(glob.glob(os.path.join(cfg.save_dir, 'step_*.pt')))
if latest_ckpt:
    report_path = generate_report(latest_ckpt[-1])
    drive_report = os.path.join(cfg.log_dir, os.path.basename(report_path))
    os.makedirs(cfg.log_dir, exist_ok=True)
    shutil.copy2(report_path, drive_report)
    print(f'Report copied to {drive_report}')
    from IPython.display import HTML, display
    with open(report_path, 'r') as f:
        display(HTML(f.read()))
else:
    print('No checkpoints found')